# GOTOken oracle

Reference values from the HuggingFace `SmolLM2-135M` model for each step's checkpoint,
and comparisons against the BASIC engine. All logic lives in `oracle.py`; this notebook
is the interactive front-end. Run `./build.sh` in the repo root first if you want the
BASIC comparisons.

In [1]:
from oracle import *
tok = load_tokenizer()
model = load_model()
model.config

/Users/bmuskalla/git/GOTOken/export/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/272 [00:00<01:35,  2.84it/s]

Loading weights:  64%|██████▍   | 174/272 [00:00<00:00, 478.40it/s]

Loading weights:  96%|█████████▋| 262/272 [00:00<00:00, 427.67it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 372.73it/s]

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dtype": "float32",
  "eos_token_id": 0,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.041666666666666664,
  "intermediate_size": 1536,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_parameters": {
    "rope_theta": 100000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0",
  "use_cache": true,
  "vocab_size": 49152
}

## Step 2: embedding row -> tied output head

No norm, no attention, no FFN. With tied weights every logit is the dot product of the
input token's embedding with one vocab row, so the "prediction" is simply the nearest
embeddings. Note what the top-5 for ` cat` looks like.

In [2]:
for i in tok.encode("The cat sat", add_special_tokens=False):
    print(i, repr(tok.convert_ids_to_tokens(i)))

504 'The'
2644 'Ġcat'
2643 'Ġsat'


In [3]:
_ = print_step2(model, tok, 2644)

token 2644 = 'Ġcat'
logit 0 0.133899
logit 1 1.771975
logit 2 1.769763
logit 3 1.899459
logit 4 1.86695
top 1 id 2644 logit 3.985926   'Ġcat'
top 2 id 9786 logit 3.531259   'cat'
top 3 id 40578 logit 3.432986   'cats'
top 4 id 27772 logit 3.333802   'Cat'
top 5 id 6 logit 3.297556   '<filename>'


In [4]:
compare_step2(model, tok, 2644)


compare BASIC vs oracle
  logit 0: basic 0.133899  oracle 0.133899  diff 6.12e-09  ok
  logit 1: basic 1.771975  oracle 1.771975  diff 4.50e-07  ok
  logit 2: basic 1.769764  oracle 1.769763  diff 7.33e-07  ok
  logit 3: basic 1.899459  oracle 1.899459  diff 3.95e-07  ok
  logit 4: basic 1.86695  oracle 1.86695  diff 1.35e-07  ok
  top 1: basic id 2644 3.985922  oracle id 2644 3.985926  diff 3.90e-06  ok
  top 2: basic id 9786 3.531259  oracle id 9786 3.531259  diff 1.81e-07  ok
  top 3: basic id 40578 3.432987  oracle id 40578 3.432986  diff 8.10e-07  ok
  top 4: basic id 27772 3.333801  oracle id 27772 3.333802  diff 8.67e-07  ok
  top 5: basic id 6 3.297555  oracle id 6 3.297556  diff 7.97e-07  ok
PASS


True

## Step 3: the kernels in isolation

RMSNorm and MatMul on real layer-0 weights, fed the embedding row of ` cat`. The float64
result is the reference; the pass criterion is `|basic - ref| <= 1e-5 + 1e-5 * |ref|`.

The second table is the lesson: the same fp32 matmul computed four ways agrees with
BASIC bit for bit only when the accumulation order is replayed exactly. Nothing else
does, not even torch, and none of them is "wrong".

In [5]:
compare_step3(model, tok, 2644)


compare BASIC vs float64 oracle, token 2644 = 'Ġcat'
  criterion: |basic - ref| <= 1e-05 + 1e-05 * |ref|
  x        n= 576  max abs err 0.00e+00  max rel err 0.00e+00  bit-exact vs float64 576/576  ok
  rmsnorm  n= 576  max abs err 4.84e-07  max rel err 5.38e-07  bit-exact vs float64 0/576  ok
  wq       n= 576  max abs err 3.90e-06  max rel err 8.49e-06  bit-exact vs float64 0/576  ok
  wk       n= 192  max abs err 5.46e-06  max rel err 9.77e-06  bit-exact vs float64 0/192  ok
  w1       n=1536  max abs err 1.02e-06  max rel err 1.79e-05  bit-exact vs float64 0/1536  ok

same fp32 matmul (wq), different accumulation: bits identical to BASIC
  sequential fp32, round mul then add    576/576  max diff 0.00e+00
  sequential fp32, fused multiply-add    243/576  max diff 9.54e-07
  torch fp32 matmul (BLAS order)          61/576  max diff 1.91e-06
  float64 reference rounded to fp32       26/576  max diff 3.81e-06

FLOPs per token: matmul ~2.69e+08, rmsnorm ~1.05e+05 -> matmul share 99.9608

True

## Step 4: one transformer layer

Layer 0 run over a short token sequence, token *i* at position *i*. The reference for
the layer output is a forward hook on `model.model.layers[0]` at the last position; the
references for q and k after RoPE are an independent numpy implementation of the
interleaved rotation.

With a single token, attention is trivial (one score, softmax gives 1, the output is
just v), so the sequence is what actually exercises the attention loop, the GQA head
mapping, and RoPE making scores depend on the distance between positions.

In [6]:
compare_step4(model, tok, tok.encode("The cat sat on the", add_special_tokens=False))


compare BASIC layer 0 vs HF forward hook over 5 tokens ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe'], output at the last position
  criterion: |basic - ref| <= 0.0001 + 0.0001 * |ref|
  q_rope   n= 576  max abs err 4.04e-06  max rel err 9.03e-06  bit-exact vs float64 0/576  ok
  k_rope   n= 192  max abs err 4.52e-06  max rel err 6.17e-06  bit-exact vs float64 0/192  ok
  layer0   n= 576  max abs err 5.72e-06  max rel err 1.13e-04  bit-exact vs float64 23/576  ok
  residual stream at the last position: |x_in| rms 0.1159 -> |x_out| rms 1.2929
PASS


True

## Step 5: full forward and greedy decoding

All 30 layers, final norm, classifier. First the logits at the last prompt position are
compared with the real model, then greedy decoding is compared token for token against
`model.generate(do_sample=False)`.

`margin` is the gap between the best and second-best logit at each step: the numerical
headroom the greedy choice had. The BASIC engine re-forwards the whole sequence for every
new token (no KV cache yet), so watch the seconds per step grow linearly.

In [7]:
compare_step5(model, tok, tok.encode("The cat sat on the", add_special_tokens=False), 8)


full forward over ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe']: logits at the last position
  criterion: |basic - ref| <= 0.001 + 0.001 * |ref|
  logit 0: basic 7.290724  oracle 7.290715  diff 9.3e-06  ok
  logit 1: basic -2.707315  oracle -2.707330  diff 1.5e-05  ok
  logit 2: basic -2.664011  oracle -2.664025  diff 1.4e-05  ok
  logit 3: basic -4.295832  oracle -4.295849  diff 1.7e-05  ok
  logit 4: basic -3.879560  oracle -3.879569  diff 9.1e-06  ok
  top 1: basic id 4463 18.069260  oracle id 4463 18.069252  diff 8.0e-06  'Ġbed'  ok
  top 2: basic id 5595 17.590830  oracle id 5595 17.590834  diff 3.7e-06  'Ġedge'  ok
  top 3: basic id 5700 17.282160  oracle id 5700 17.282162  diff 1.7e-06  'Ġwindow'  ok
  top 4: basic id 3252 17.260630  oracle id 3252 17.260614  diff 1.6e-05  'Ġtable'  ok
  top 5: basic id 3187 16.982280  oracle id 3187 16.982281  diff 7.3e-07  'Ġtree'  ok



greedy decode without KV cache, 8 tokens
  step 0: basic   4463 'Ġbed'         oracle   4463  logit diff 8.0e-06  margin 0.478  5.3s  ok
  step 1: basic     28 ','            oracle     28  logit diff 1.3e-05  margin 0.362  6.1s  ok
  step 2: basic    284 'Ġand'         oracle    284  logit diff 2.1e-06  margin 0.870  7.0s  ok
  step 3: basic    260 'Ġthe'         oracle    260  logit diff 3.0e-06  margin 0.344  8.1s  ok
  step 4: basic   2644 'Ġcat'         oracle   2644  logit diff 3.9e-06  margin 0.189  9.4s  ok
  step 5: basic   2643 'Ġsat'         oracle   2643  logit diff 1.6e-05  margin 0.937  10.4s  ok
  step 6: basic    335 'Ġon'          oracle    335  logit diff 1.4e-06  margin 2.667  13.1s  ok
  step 7: basic    260 'Ġthe'         oracle    260  logit diff 1.2e-05  margin 3.695  12.9s  ok
  smallest margin 0.189
  text: 'The cat sat on the' -> ' bed, and the cat sat on the'
  BASIC: 68 forwards in 72.2s, 0.11 tok/s
PASS


True

## Step 6: the KV cache

The same greedy run twice: re-forwarding the whole prefix for every token (step 5) and
forwarding only the new token with earlier k/v rows kept (step 6). The chosen logits must
be bit-identical, because a cached row is exactly the number the recompute would have
produced again. The speedup at step *s* is `n_prompt + s`: the cache turns per-token
cost from O(n) forwards into 1, so the whole generation goes from O(n²) to O(n).

In [8]:
compare_step6(model, tok, tok.encode("The cat sat on the", add_special_tokens=False), 8, 24)


KV cache vs re-forwarding the prefix, prompt ['The', 'Ġcat', 'Ġsat', 'Ġon', 'Ġthe'], 8 tokens
  step  seqlen  token            HF  same id  same bits   no-cache fwds  secs   cache fwds  secs   speedup
     0       5  'Ġbed'           4463      yes        yes               5    5.0            5   4.93     1.0x
     1       6  ','                28      yes        yes               6    5.9            1   0.98     6.1x
     2       7  'Ġand'            284      yes        yes               7    6.8            1   0.97     7.0x
     3       8  'Ġthe'            260      yes        yes               8    7.8            1   0.98     8.0x
     4       9  'Ġcat'           2644      yes        yes               9    8.8            1   0.98     9.0x
     5      10  'Ġsat'           2643      yes        yes              10   11.3            1   0.97    11.6x
     6      11  'Ġon'             335      yes        yes              11   11.3            1   0.97    11.6x
     7      12  'Ġthe'      


cached run of 24 tokens: per-token secs after prefill first 0.98, middle 1.19, last 0.99 at seqlen 28
  text: 'The cat sat on the' -> ' bed, and the cat sat on the bed.\n\nThe cat sat on the bed.\n\nThe cat sat'


  matches HF greedy for 24/24 tokens
PASS


np.True_

## Step 7: the tokenizer

`bpe_ref.py` is the tokenizer written in Python against nothing but `tokenizer.bin`, validated
against HuggingFace first; `src/tokenizer.bm` is the same algorithm in BASIC. This cell runs
the BASIC encoder over the whole corpus (hand-written edge cases plus 2000 random strings over
many scripts) and compares ids with HuggingFace, then round-trips a few strings through the
BASIC decoder.

In [9]:
import bpe_ref
bpe_ref.validate()          # the Python reference vs HuggingFace
compare_step7(tok)          # the BASIC engine vs HuggingFace

2060/2060 strings match HuggingFace (60 hand-written, 2000 random)



tokenizer: BASIC vs HuggingFace on 2060 strings (60 hand-written, 2000 random): 2060 match
  BASIC encoded 2060 strings in 0.07s


  decode 'Hello world'                                      ok


  decode 'The quick brown fox jumps over the lazy dog.'     ok


  decode '  leading spaces and 12345 digits'                ok


  decode 'Ünïcödé ✓ 日本語 🚀 naïve café'                       ok


  decode 'def main():\n    return 0\n'                      ok


  decode 'Hello\n\nWorld'                                   ok
PASS


True

## Step 8: the sampler

Logits become a distribution (`softmax(logits / temperature)`), the distribution is trimmed
(top-k, top-p), and one random number picks from what is left. The random generator is
run.c's xorshift64*, seeded, so a sampled run is exactly as reproducible as a greedy one:
the reference sampler here, fed HuggingFace's logits and the same seed, must produce the
same coins and the same tokens as the BASIC engine.

The grid at the end is the point of the step: the same prompt at four temperatures.

In [10]:
compare_step8(model, tok, tok.encode("Once upon a time", add_special_tokens=False), 8)


temperature 0 vs HF greedy: identical  ', there was a little girl named Lily'


temperature 0.8 top-p 0.9 top-k 0 seed 42: BASIC vs reference sampler 8/8 tokens  ', back in 1865'
    step 0: coin 0.3390852 vs 0.3390852  candidates     2  basic     28 ','            ref     28  ok
    step 1: coin 0.7822558 vs 0.7822558  candidates    23  basic   1056 'Ġback'        ref   1056  ok
    step 2: coin 0.7901370 vs 0.7901370  candidates     2  basic    281 'Ġin'          ref    281  ok
    step 3: coin 0.9440426 vs 0.9440426  candidates     2  basic    216 'Ġ'            ref    216  ok
    step 4: coin 0.7643936 vs 0.7643936  candidates     2  basic     33 '1'            ref     33  ok
    step 5: coin 0.8357399 vs 0.8357399  candidates     3  basic     40 '8'            ref     40  ok
    step 6: coin 0.2042197 vs 0.2042197  candidates     9  basic     38 '6'            ref     38  ok
    step 7: coin 0.4398116 vs 0.4398116  candidates     9  basic     37 '5'            ref     37  ok


temperature 1.0 top-p 1.0 top-k 40 seed 7: BASIC vs reference sampler 8/8 tokens  ' in Germany, there lived a small town'
    step 0: coin 0.8202466 vs 0.8202466  candidates    40  basic    281 'Ġin'          ref    281  ok
    step 1: coin 0.9282901 vs 0.9282901  candidates    40  basic   4521 'ĠGermany'     ref   4521  ok
    step 2: coin 0.0893496 vs 0.0893496  candidates    40  basic     28 ','            ref     28  ok
    step 3: coin 0.1076274 vs 0.1076274  candidates    40  basic    665 'Ġthere'       ref    665  ok
    step 4: coin 0.3745358 vs 0.3745358  candidates    40  basic   4161 'Ġlived'       ref   4161  ok
    step 5: coin 0.4072740 vs 0.4072740  candidates    40  basic    253 'Ġa'           ref    253  ok
    step 6: coin 0.8528833 vs 0.8528833  candidates    40  basic   1165 'Ġsmall'       ref   1165  ok
    step 7: coin 0.1707058 vs 0.1707058  candidates    40  basic   3102 'Ġtown'        ref   3102  ok

prompt 'Once upon a time', 8 tokens, top-p 0.9, three seeds p

  temperature 0.2: avg candidates     1  distinct 1/3
      ', there was a little girl named Lily'
      ', there was a little girl named Lily'
      ', there was a little girl named Lily'


  temperature 0.7: avg candidates     4  distinct 3/3
      ', in a land far away, there'
      ', there was a little girl named Lily'
      ', there was a wise old owl named'


  temperature 1.0: avg candidates   137  distinct 3/3
      ', it was quite a distant hill country'
      ', in a faraway land called Australia,'
      ' in a small village,\nlived a'


  temperature 1.5: avg candidates  5149  distinct 3/3
      ', Jeremy Staten was dismissed out on'
      ' in Asia there was a bad badd'
      '—and one organismesDate<|endoftext|>It'
PASS


True